# Hypothesis Test

## Hypothesis Test 1: Change in Mean Agility



$$
H_0: \mu_B \leq \mu_A
$$


$$
H_1: \mu_B > \mu_A
$$


In [60]:
import pandas as pd
import numpy as np
import plotly

In [61]:
from scipy.stats import shapiro

def check_normality(data, group_name):
    
    stat, p_value = shapiro(data)

    if p_value > 0.05:
        print(f"{group_name}: Data is normal.")
    else:
        print(f"{group_name}: Data is not normal.")


import plotly.express as px

def plot_histogram(data, title, x_label, color="#636EFA", bins=20):

    fig = px.histogram(
        x=data,
        nbins=bins,
        title=title,
        labels={"x": x_label, "count": "Frequency"},
        template="plotly_white",
        color_discrete_sequence=[color])

    fig.update_layout(
        title_x=0.5,
        xaxis_title=x_label,
        yaxis_title="Frequency",
        bargap=0.1)

    fig.show()

### Data Preparation for Hypothesis Testing


In [62]:
df = pd.read_csv("rosters_cleaned_and_engineered.csv")
df.head()

,Season,Team,No.,Player,Pos,Ht,Wt,Birth Date,Birth,Exp,...,Wt_KG,Exp_Numeric,Birth_Date_DT,No_Numeric,Agility_Metric,BMI,Rookie_Flag,Roster_Exp_Tier,Estimated_Age,Innate_Ability_Roster
0,1980,ATL,50,John Brown,SF,7-Jun,220,14-Dec-51,deDE,6,...,99.79,6,12/14/1951,50.0,2.0108,24.78,0,Veteran,29,0.2069
1,1980,ATL,14,Charlie Criss,PG,8-May,165,6-Nov-48,usUS,2,...,74.84,2,11/6/1948,14.0,2.3079,25.09,0,Developing,32,0.0625
2,1980,ATL,22,John Drew,SF,6-Jun,205,30-Sep-54,usUS,5,...,92.99,5,9/30/1954,22.0,2.1306,23.69,0,Veteran,26,0.1923
3,1980,ATL,25,Terry Furlow,SG,4-Jun,190,18-Oct-54,usUS,3,...,86.18,3,10/18/1954,25.0,2.2400,23.13,0,Developing,26,0.1154
4,1980,ATL,21,Jack Givens,SF,5-Jun,205,21-Sep-56,usUS,1,...,92.99,1,9/21/1956,21.0,2.1032,24.31,0,Developing,24,0.0417


In [63]:
seasons = [2020, 2021, 2022, 2023]
df_alternative = df[df["Season"].isin(seasons)].copy()
top_20 = df_alternative.groupby("Season", sort=True).head(20)

top_20.head()

,Season,Team,No.,Player,Pos,Ht,Wt,Birth Date,Birth,Exp,...,Wt_KG,Exp_Numeric,Birth_Date_DT,No_Numeric,Agility_Metric,BMI,Rookie_Flag,Roster_Exp_Tier,Estimated_Age,Innate_Ability_Roster
683,2020,ATL,95,DeAndre' Bembry,SG,5-Jun,210,4-Jul-94,usUS,3,...,95.25,3,7/4/1994,95.0,2.0533,24.90,0,Developing,26,0.1154
684,2020,ATL,4,Charlie Brown Jr.,SG,6-Jun,199,2-Feb-97,usUS,R,...,90.26,0,2/2/1997,4.0,2.1950,23.00,1,Rookie,23,0.0000
685,2020,ATL,15,Vince Carter,PF,6-Jun,220,26-Jan-77,usUS,21,...,99.79,21,1/26/1977,15.0,1.9854,25.42,0,Veteran,43,0.4884
686,2020,ATL,20,John Collins,PF,9-Jun,226,23-Sep-97,usUS,2,...,102.51,2,9/23/1997,20.0,2.0070,24.22,0,Developing,23,0.0870
687,2020,ATL,33,Allen Crabbe,SG,5-Jun,212,9-Apr-92,usUS,6,...,96.16,6,4/9/1992,33.0,2.0339,25.14,0,Veteran,28,0.2143


In [64]:
group_A = top_20[top_20["Season"].isin([2020, 2021])]["Agility_Metric"]
group_B = top_20[top_20["Season"].isin([2022, 2023])]["Agility_Metric"]

print("Group A mean:", group_A.mean())
print("Group A standard deviation:", group_A.std())
print("------------------")
print("Group B mean:", group_B.mean())
print("Group B standard deviation:", group_B.std())

Group A mean: 2.033045
Group A standard deviation: 0.16727238641332962
------------------
Group B mean: 2.066865
Group B standard deviation: 0.1764256414150674


### Testing the Independence of Groups A and B


In [65]:
players_A = set(top_20[top_20["Season"].isin([2020, 2021])]["Player"])
players_B = set(top_20[top_20["Season"].isin([2022, 2023])]["Player"])

print("Players appearing in both groups:", len(players_A.intersection(players_B)))


Players appearing in both groups: 12


### Testing for Normality


In [66]:
check_normality(group_A, "Group A")
check_normality(group_B, "Group B")

Group A: Data is not normal.
Group B: Data is not normal.


In [72]:
plot_histogram(group_A, title="Agility Distribution - Group A",  x_label="Agility")
plot_histogram(group_B, title="Agility Distribution - Group B", x_label="Agility")

### Wilcoxon Signed-Rank Test


In [73]:
from scipy.stats import wilcoxon

n = min(len(group_A), len(group_B))

group_A = group_A.iloc[:n]
group_B = group_B.iloc[:n]

stat, p_value = wilcoxon(group_B, group_A, alternative="greater")

print("Wilcoxon Signed-Rank Test")
print("Test statistic:", stat)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

Wilcoxon Signed-Rank Test
Test statistic: 468.0
P-value: 0.2178119865112047
Fail to reject H0


## Hypothesis Test 2: Change in Players’ Innate Ability

**H₀:** توانایی ذاتی بازیکنان تیم‌های قهرمان در فصل‌های ۲۰۲۵ و ۲۰۲۶ نسبت به فصل‌های ۲۰۲۳ و ۲۰۲۴ افزایش نیافته است.

**H₁:** توانایی ذاتی بازیکنان تیم‌های قهرمان در فصل‌های ۲۰۲۵ و ۲۰۲۶ نسبت به فصل‌های ۲۰۲۳ و ۲۰۲۴ افزایش یافته است.


### Data Preparation: Comparing Innate Ability Across Seasons


In [11]:
import pandas as pd
import numpy as np
from scipy import stats

rosters = pd.read_csv("rosters_cleaned_and_engineered.csv")
champions = pd.read_csv("cross_champions_vs_top_players.csv")

In [12]:
rosters.head()

,Season,Team,No.,Player,Pos,Ht,Wt,Birth Date,Birth,Exp,...,Wt_KG,Exp_Numeric,Birth_Date_DT,No_Numeric,Agility_Metric,BMI,Rookie_Flag,Roster_Exp_Tier,Estimated_Age,Innate_Ability_Roster
0,1980,ATL,50,John Brown,SF,7-Jun,220,14-Dec-51,deDE,6,...,99.79,6,12/14/1951,50.0,2.0108,24.78,0,Veteran,29,0.2069
1,1980,ATL,14,Charlie Criss,PG,8-May,165,6-Nov-48,usUS,2,...,74.84,2,11/6/1948,14.0,2.3079,25.09,0,Developing,32,0.0625
2,1980,ATL,22,John Drew,SF,6-Jun,205,30-Sep-54,usUS,5,...,92.99,5,9/30/1954,22.0,2.1306,23.69,0,Veteran,26,0.1923
3,1980,ATL,25,Terry Furlow,SG,4-Jun,190,18-Oct-54,usUS,3,...,86.18,3,10/18/1954,25.0,2.2400,23.13,0,Developing,26,0.1154
4,1980,ATL,21,Jack Givens,SF,5-Jun,205,21-Sep-56,usUS,1,...,92.99,1,9/21/1956,21.0,2.1032,24.31,0,Developing,24,0.0417


In [13]:
champions.head()

,Season_Year,Champion_Team,Top15_Avg_Height_CM,Top15_Avg_Age,Top15_Avg_Agility,Top15_Leading_Scorer,Top15_Max_PTS
0,2026,New York Knicks,198.0,29.5,1.9958,Luka Dončić,2143.0
1,2025,Oklahoma City Thunder,198.5,28.1,2.0623,Shai Gilgeous-Alexander,2484.0
2,2024,Boston Celtics,199.5,28.6,1.9965,Luka Dončić,2370.0
3,2023,Denver Nuggets,198.5,26.5,2.0235,Jayson Tatum,2225.0
4,2022,Golden State Warriors,201.7,27.2,1.9795,Trae Young,2155.0


In [15]:
print(sorted(rosters["Season"].dropna().unique()))
print(sorted(champions["Season_Year"].dropna().unique()))

[1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]
[1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [ ]:
print("Seasons in rosters:")
print(sorted(rosters["Season"].unique()))
print("\nSeasons in champions:")
print(sorted(champions["Season_Year"].unique()))

Seasons in rosters:
[1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027]

Seasons in champions:
[1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [50]:
team_names = {
    "ATL": "Atlanta Hawks",
    "BOS": "Boston Celtics",
    "NJN": "New Jersey Nets",
    "CHA": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards",
    "BKN": "Brooklyn Nets",
    "SEA": "Seattle SuperSonics",
    "VAN": "Vancouver Grizzlies",
    "NOH": "New Orleans Hornets",
    "NOK": "New Orleans/Oklahoma City Hornets",
    "WSB": "Washington Bullets",
    "KCK": "Kansas City Kings",
    "SDC": "San Diego Clippers",
    "BUF": "Buffalo Braves",
    "PHO": "Phoenix Suns",
    "CHH": "Charlotte Hornets",
    "UTH": "Utah Jazz"
}

rosters["Team_Full"] = rosters["Team"].map(team_names)


In [ ]:
merged = rosters.merge(champions[["Season_Year", "Champion_Team"]].drop_duplicates(),
    left_on="Season", right_on="Season_Year", how="inner")

champion_players = merged[merged["Team_Full"] == merged["Champion_Team"]].copy()

champion_players.groupby("Season").agg(
    Number_of_Players=("Player", "nunique"),
    Mean_Innate_Ability=("Innate_Ability_Roster", "mean"))

,Number_of_Players,Mean_Innate_Ability
Season,,
1980,15,0.124540
1981,12,0.104567
1982,14,0.139857
1983,14,0.110621
1984,12,0.158317
1985,14,0.167450
1986,13,0.172815
1987,13,0.150838
1988,15,0.170893


In [27]:
print(sorted(champion_players["Season"].unique())[-4:])

[2023, 2024, 2025, 2026]


In [28]:
player_seasons = champion_players.groupby("Player")["Season"].nunique()
print(player_seasons.value_counts().sort_index())

Season
1    379
2     77
3     36
4     13
5      9
6      2
7      1
Name: count, dtype: int64


In [51]:
recent_seasons = [2025, 2026]
previous_seasons = [2023, 2024]

recent_players = champion_players[champion_players["Season"].isin(recent_seasons)].copy()
previous_players = champion_players[champion_players["Season"].isin(previous_seasons)].copy()

### Testing the Independence of Innate Ability Across Seasons

In [52]:
recent_names = set(recent_players["Player"])
previous_names = set(previous_players["Player"])

print("Number of recent players:", len(recent_names))
print("Number of previous players", len(previous_names))
print("Number of common players:", len(recent_names.intersection(previous_names)))

Number of recent players: 38
Number of previous players 37
Number of common players: 0


### Testing the Normality of Innate Ability Across Seasons

In [ ]:
check_normality(recent_players["Innate_Ability_Roster"], "Recent Seasons")
check_normality(previous_players["Innate_Ability_Roster"], "Previous Seasons")

Recent Seasons: Data is not normal.
Previous Seasons: Data is not normal.


In [56]:
print(plot_histogram(recent_players["Innate_Ability_Roster"],
    title="Innate Ability - Recent Seasons",
    x_label="Innate Ability"))

print(plot_histogram(previous_players["Innate_Ability_Roster"],
    title="Innate Ability - Previous Seasons",
    x_label="Innate Ability"))


None


None


### Mann–Whitney U Test: Comparing Innate Ability Across Seasons

In [ ]:

from scipy.stats import mannwhitneyu

recent_ability = recent_players["Innate_Ability_Roster"]
previous_ability = previous_players["Innate_Ability_Roster"]

statistic, p_value = mannwhitneyu(recent_ability, previous_ability, alternative="greater")

print("Mann-Whitney U statistic:", statistic)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

Mann-Whitney U statistic: 557.5
P-value: 0.956671164336633
Fail to reject H0: There is not enough evidence of an increase in innate ability.
